# Cillian AI Studio
10 Open-Source Modelle gratis auf Google Colab T4 GPU.

| # | Modell | Typ | ca. Dauer |
|---|--------|-----|-----------|
| 1 | LTX 2.3 | Text→Video | ~2 min |
| 2 | LTX 2.3 | **Image→Video** | ~2 min |
| 3 | Wan 2.1 | Text→Video | ~4 min |
| 4 | Wan 2.1 | **Image→Video** | ~4 min |
| 5 | Flux Dev | Text→Bild (Qualitaet) | ~80s |
| 6 | Flux Schnell | Text→Bild (schnell) | ~15s |
| 7 | SDXL Lightning | Text→Bild | ~30s |
| 8 | PixArt-Sigma | Text→Bild (klein) | ~20s |
| 9 | SD 3.5 Medium | Text→Bild | ~40s |
| 10 | Whisper Large V3 | Sprache→Text | ~5s |
| 11 | Qwen2-Audio | Audio verstehen | ~10s |
| 12 | Kokoro TTS | Text→Sprache | ~2s |

Alle Zellen der Reihe nach ausfuehren. Am Ende bekommst du eine API-URL.

**API Key:** `ironclaw-video-2026`

## 1. GPU Check

In [ ]:
!nvidia-smi
import torch
print(f'\nCUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
assert torch.cuda.is_available(), 'KEINE GPU! Gehe zu Laufzeit > Laufzeittyp aendern > T4 GPU'

## 2. ComfyUI + Nodes installieren (~2 min)

In [ ]:
import os, subprocess

if not os.path.exists('/content/ComfyUI'):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -r requirements.txt -q
else:
    %cd /content/ComfyUI
    print('ComfyUI already installed')

# Custom Nodes
nodes = {
    'ComfyUI-GGUF': 'https://github.com/city96/ComfyUI-GGUF.git',
    'ComfyUI-LTXVideo': 'https://github.com/Lightricks/ComfyUI-LTXVideo.git',
    'ComfyUI-KJNodes': 'https://github.com/kijai/ComfyUI-KJNodes.git',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    # NEU v3: Frame Interpolation (RIFE) — fluessigere Videos
    'ComfyUI-Frame-Interpolation': 'https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git',
    # NEU v3: TeaCache — 1.5-3x schnellere Video-Generierung
    'ComfyUI-TeaCache': 'https://github.com/welltop-cn/ComfyUI-TeaCache.git',
    # NEU v3: WAN 2.2 Video Wrapper (Kijai)
    'ComfyUI-WanVideoWrapper': 'https://github.com/kijai/ComfyUI-WanVideoWrapper.git',
    # NEU v3.2: ControlNet + Pose Extraction (fuer Video Dance Transfer)
    'comfyui_controlnet_aux': 'https://github.com/Fannovel16/comfyui_controlnet_aux.git',
    'ComfyUI-Advanced-ControlNet': 'https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet.git',
}
for name, url in nodes.items():
    d = f'/content/ComfyUI/custom_nodes/{name}'
    if not os.path.exists(d):
        !git clone --depth 1 {url} {d}
        req = os.path.join(d, 'requirements.txt')
        if os.path.exists(req):
            !pip install -r {req} -q 2>/dev/null

!pip install gguf huggingface_hub hf_transfer -q

# T4 patch: float16 support (T4 has no bfloat16)
for pyfile in ['comfy/model_management.py', 'comfy/sd.py']:
    fp = f'/content/ComfyUI/{pyfile}'
    if os.path.exists(fp):
        with open(fp, 'r') as f: c = f.read()
        if 'working_dtypes = [torch.bfloat16]' in c:
            c = c.replace('working_dtypes = [torch.bfloat16]', 'working_dtypes = [torch.bfloat16, torch.float16]')
            with open(fp, 'w') as f: f.write(c)
            print(f'T4 patch: {pyfile}')

# NEU v3.3: LoRA Training (kohya sd-scripts)
lora_dir = '/content/sd-scripts'
if not os.path.exists(lora_dir):
    !git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git {lora_dir}
    !pip install -r {lora_dir}/requirements.txt -q 2>/dev/null
    !pip install bitsandbytes prodigyopt lion-pytorch -q
    print('kohya sd-scripts installed')
else:
    print('kohya sd-scripts already installed')

print('\nSetup complete! (v3.3: +RIFE +TeaCache +WanVideoWrapper +LoRA Training)')

## 3. Google Drive + Download Helper

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CACHE = '/content/drive/MyDrive/ai-models'
os.makedirs(CACHE, exist_ok=True)

DIRS = {}
for d in ['unet', 'clip', 'clip_vision', 'vae', 'loras', 'checkpoints']:
    p = f'/content/ComfyUI/models/{d}'
    os.makedirs(p, exist_ok=True)
    DIRS[d] = p

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download, snapshot_download
import shutil

def dl(repo, filename, target_dir):
    safe = filename.replace('/', '__')
    cache = os.path.join(CACHE, safe)
    target = os.path.join(target_dir, os.path.basename(filename))
    if os.path.exists(target):
        print(f'  OK: {os.path.basename(filename)}')
        return
    if os.path.exists(cache):
        os.symlink(cache, target)
        print(f'  Cache: {os.path.basename(filename)}')
        return
    print(f'  Downloading: {os.path.basename(filename)}...')
    p = hf_hub_download(repo_id=repo, filename=filename, local_dir='/tmp/hf_dl')
    shutil.move(p, cache)
    os.symlink(cache, target)
    print(f'  Done: {os.path.basename(filename)}')

print('Ready!')

## 4. Video-Modelle herunterladen

In [ ]:
# ═══ LTX 2.3 (bleibt gleich) ═══
print('=== LTX 2.3 (T2V + I2V, ~13 GB) ===')
dl('unsloth/LTX-2.3-GGUF', 'distilled/ltx-2.3-22b-distilled-Q4_K_S.gguf', DIRS['unet'])
dl('Kijai/LTX2.3_comfy', 'vae/LTX23_video_vae_bf16.safetensors', DIRS['vae'])
dl('Kijai/LTX2.3_comfy', 'vae/LTX23_audio_vae_bf16.safetensors', DIRS['vae'])
dl('Kijai/LTX2.3_comfy', 'loras/ltx-2.3-22b-distilled-lora-dynamic_fro09_avg_rank_105_bf16.safetensors', DIRS['loras'])

# ═══ WAN 2.2 MoE (NEU — ersetzt 2.1) ═══
print('\n=== Wan 2.2 T2V MoE-14B (~10 GB) ===')
dl('QuantStack/Wan2.2-T2V-A14B-GGUF', 'HighNoise/Wan2.2-T2V-A14B-HighNoise-Q4_K_S.gguf', DIRS['unet'])
dl('QuantStack/Wan2.2-T2V-A14B-GGUF', 'LowNoise/Wan2.2-T2V-A14B-LowNoise-Q4_K_S.gguf', DIRS['unet'])

print('\n=== Wan 2.2 I2V MoE-14B (~10 GB) ===')
dl('QuantStack/Wan2.2-I2V-A14B-GGUF', 'HighNoise/Wan2.2-I2V-A14B-HighNoise-Q4_K_S.gguf', DIRS['unet'])
dl('QuantStack/Wan2.2-I2V-A14B-GGUF', 'LowNoise/Wan2.2-I2V-A14B-LowNoise-Q4_K_S.gguf', DIRS['unet'])

print('\n=== Wan 2.2 TI2V 5B — leichtes Text+Image→Video (~4 GB) ===')
dl('QuantStack/Wan2.2-TI2V-5B-GGUF', 'Wan2.2-TI2V-5B-Q4_K_S.gguf', DIRS['unet'])

# ═══ WAN 2.1 behalten als Fallback ═══
print('\n=== Wan 2.1 T2V (Fallback, ~10 GB) ===')
dl('city96/Wan2.1-T2V-14B-gguf', 'wan2.1-t2v-14b-Q4_K_S.gguf', DIRS['unet'])

print('\n=== Wan 2.1 I2V (Fallback, ~10 GB) ===')
dl('city96/Wan2.1-I2V-14B-480p-gguf', 'wan2.1-i2v-14b-480p-Q4_K_S.gguf', DIRS['unet'])

# ═══ Shared Video Components ═══
print('\n=== Shared Video Components ===')
dl('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/vae/wan_2.1_vae.safetensors', DIRS['vae'])
dl('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', DIRS['clip'])
dl('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/clip_vision/clip_vision_h.safetensors', DIRS['clip_vision'])

# ═══ 4x-UltraSharp Upscaler (NEU) ═══
print('\n=== 4x-UltraSharp Upscaler (~67 MB) ===')
import os, urllib.request
upscale_dir = os.path.join('/content/ComfyUI/models/upscale_models')
os.makedirs(upscale_dir, exist_ok=True)
us_path = os.path.join(upscale_dir, '4x-UltraSharp.pth')
if not os.path.exists(us_path):
    # Drive-Cache pruefen
    drive_us = '/content/drive/MyDrive/ai-models/4x-UltraSharp.pth'
    if os.path.exists(drive_us):
        import shutil; shutil.copy2(drive_us, us_path)
        print('Copied from Drive cache')
    else:
        urllib.request.urlretrieve(
            'https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth',
            us_path)
        # Cache auf Drive
        os.makedirs(os.path.dirname(drive_us), exist_ok=True)
        import shutil; shutil.copy2(us_path, drive_us)
        print('Downloaded + cached on Drive')
else:
    print('4x-UltraSharp already installed')

print('\nVideo models + upscaler ready! (v3: WAN 2.2 MoE + UltraSharp)')

## 5. Bild-Modelle herunterladen

In [ ]:
print('=== Flux Dev GGUF (~7 GB) ===')
dl('city96/FLUX.1-dev-gguf', 'flux1-dev-Q4_K_S.gguf', DIRS['unet'])

print('\n=== Flux Schnell GGUF (~7 GB) ===')
dl('city96/FLUX.1-schnell-gguf', 'flux1-schnell-Q4_K_S.gguf', DIRS['unet'])

print('\n=== Shared Flux Components ===')
dl('comfyanonymous/flux_text_encoders', 'clip_l.safetensors', DIRS['clip'])
dl('comfyanonymous/flux_text_encoders', 't5xxl_fp8_e4m3fn.safetensors', DIRS['clip'])
dl('ffxvs/vae-flux', 'ae.safetensors', DIRS['vae'])

print('\n=== SDXL Lightning (~7 GB) ===')
dl('ByteDance/SDXL-Lightning', 'sdxl_lightning_4step_unet.safetensors', DIRS['unet'])

print('\n=== SD 3.5 Medium GGUF (~5 GB) ===')
dl('city96/stable-diffusion-3.5-medium-gguf', 'sd3.5_medium-Q4_K_S.gguf', DIRS['unet'])
dl('Comfy-Org/stable-diffusion-3.5-fp8', 'text_encoders/clip_g.safetensors', DIRS['clip'])
dl('Comfy-Org/stable-diffusion-3.5-fp8', 'text_encoders/clip_l.safetensors', DIRS['clip'])

print('\n=== Fluxed Up v7.1 NSFW GGUF (~6.5 GB) ===')
import subprocess as _sp
fluxedup_path = os.path.join(DIRS['unet'], 'fluxedupV71_q4ks.gguf')
if os.path.exists(fluxedup_path) and os.path.getsize(fluxedup_path) > 1000000:
    print('  OK: fluxedupV71_q4ks.gguf')
else:
    cache_path = os.path.join(CACHE, 'fluxedupV71_q4ks.gguf') if CACHE else None
    if cache_path and os.path.exists(cache_path) and os.path.getsize(cache_path) > 1000000:
        import shutil; shutil.copy(cache_path, fluxedup_path); print('  Cache (Drive): fluxedupV71_q4ks.gguf')
    else:
        print('  Downloading: fluxedupV71_q4ks.gguf...')
        _sp.run(['wget', '-q', 'https://civitai.com/api/download/models/2625846?token=820f5f6ffe57e676e8a19a3e603b72a8', '-O', fluxedup_path], check=True)
        if cache_path:
            import shutil; shutil.copy(fluxedup_path, cache_path)
        print('  Done: fluxedupV71_q4ks.gguf')

print('\n=== Juggernaut XL Ragnarok (~6.8 GB) ===')
import subprocess as _sp
jugg_path = os.path.join(DIRS['checkpoints'], 'juggernautXL_ragnarok.safetensors')
if os.path.exists(jugg_path) and os.path.getsize(jugg_path) > 1000000:
    print('  OK: juggernautXL_ragnarok.safetensors')
else:
    cache_jugg = os.path.join(CACHE, 'juggernautXL_ragnarok.safetensors') if CACHE else None
    if cache_jugg and os.path.exists(cache_jugg) and os.path.getsize(cache_jugg) > 1000000:
        import shutil; shutil.copy(cache_jugg, jugg_path); print('  Cache (Drive): juggernautXL_ragnarok.safetensors')
    else:
        print('  Downloading: juggernautXL_ragnarok.safetensors...')
        _sp.run(['wget', '-q', 'https://civitai.com/api/download/models/1759168?token=820f5f6ffe57e676e8a19a3e603b72a8', '-O', jugg_path], check=True)
        if cache_jugg:
            import shutil; shutil.copy(jugg_path, cache_jugg)
        print('  Done: juggernautXL_ragnarok.safetensors')

print('\n=== RealVisXL V5.0 Lightning (~6.6 GB) ===')
realvis_path = os.path.join(DIRS['checkpoints'], 'realvisxlV50_lightning.safetensors')
if os.path.exists(realvis_path) and os.path.getsize(realvis_path) > 1000000:
    print('  OK: realvisxlV50_lightning.safetensors')
else:
    cache_rv = os.path.join(CACHE, 'realvisxlV50_lightning.safetensors') if CACHE else None
    if cache_rv and os.path.exists(cache_rv) and os.path.getsize(cache_rv) > 1000000:
        import shutil; shutil.copy(cache_rv, realvis_path); print('  Cache (Drive): realvisxlV50_lightning.safetensors')
    else:
        print('  Downloading: realvisxlV50_lightning.safetensors...')
        _sp.run(['wget', '-q', 'https://civitai.com/api/download/models/798204?token=820f5f6ffe57e676e8a19a3e603b72a8', '-O', realvis_path], check=True)
        if cache_rv:
            import shutil; shutil.copy(realvis_path, cache_rv)
        print('  Done: realvisxlV50_lightning.safetensors')



# ═══ Anime-Modelle (NEU v3.1) ═══
print('\n=== Pony Diffusion V6 XL (~7 GB) ===')
pony_dir = '/content/ComfyUI/models/checkpoints'
pony_path = os.path.join(pony_dir, 'ponyDiffusionV6XL.safetensors')
if not os.path.exists(pony_path):
    drive_pony = '/content/drive/MyDrive/ai-models/ponyDiffusionV6XL.safetensors'
    if os.path.exists(drive_pony):
        import shutil; shutil.copy2(drive_pony, pony_path); print('  Copied from Drive cache')
    else:
        dl('LyliaEngine/Pony_Diffusion_V6_XL', 'ponyDiffusionV6XL_v6StartWithThisOne.safetensors', pony_dir)
        import shutil, os as _os
        src = os.path.join(pony_dir, 'ponyDiffusionV6XL_v6StartWithThisOne.safetensors')
        if os.path.exists(src): os.rename(src, pony_path)
        _os.makedirs(os.path.dirname(drive_pony), exist_ok=True)
        if os.path.exists(pony_path) and os.path.realpath(pony_path) != os.path.realpath(drive_pony): shutil.copy2(pony_path, drive_pony); print('  Cached on Drive')
else:
    print('  Cache: ponyDiffusionV6XL.safetensors')

print('\n=== NoobAI XL v1.1 Epsilon (~7 GB) ===')
noob_path = os.path.join(pony_dir, 'NoobAI-XL-v1.1.safetensors')
if not os.path.exists(noob_path):
    drive_noob = '/content/drive/MyDrive/ai-models/NoobAI-XL-v1.1.safetensors'
    if os.path.exists(drive_noob):
        import shutil; shutil.copy2(drive_noob, noob_path); print('  Copied from Drive cache')
    else:
        dl('Laxhar/noobai-XL-1.1', 'NoobAI-XL-v1.1.safetensors', pony_dir)
        import shutil, os as _os
        _os.makedirs(os.path.dirname(drive_noob), exist_ok=True)
        if os.path.exists(noob_path) and os.path.realpath(noob_path) != os.path.realpath(drive_noob): shutil.copy2(noob_path, drive_noob); print('  Cached on Drive')
else:
    print('  Cache: NoobAI-XL-v1.1.safetensors')

print('\n=== Animagine XL 4.0 (~7 GB) ===')
anima_path = os.path.join(pony_dir, 'animagine-xl-4.0.safetensors')
if not os.path.exists(anima_path):
    drive_anima = '/content/drive/MyDrive/ai-models/animagine-xl-4.0.safetensors'
    if os.path.exists(drive_anima):
        import shutil; shutil.copy2(drive_anima, anima_path); print('  Copied from Drive cache')
    else:
        dl('cagliostrolab/animagine-xl-4.0', 'animagine-xl-4.0.safetensors', pony_dir)
        import shutil, os as _os
        _os.makedirs(os.path.dirname(drive_anima), exist_ok=True)
        if os.path.exists(anima_path) and os.path.realpath(anima_path) != os.path.realpath(drive_anima): shutil.copy2(anima_path, drive_anima); print('  Cached on Drive')
else:
    print('  Cache: animagine-xl-4.0.safetensors')



# ═══ ControlNet Modelle (NEU v3.2) ═══
print('\n=== ControlNet OpenPose SDXL (~2.5 GB) ===')
cn_dir = '/content/ComfyUI/models/controlnet'
os.makedirs(cn_dir, exist_ok=True)
cn_pose = os.path.join(cn_dir, 'controlnet-openpose-sdxl.safetensors')
if not os.path.exists(cn_pose):
    drive_cn = '/content/drive/MyDrive/ai-models/controlnet-openpose-sdxl.safetensors'
    if os.path.exists(drive_cn):
        import shutil; shutil.copy2(drive_cn, cn_pose); print('  Copied from Drive cache')
    else:
        dl('thibaud/controlnet-openpose-sdxl-1.0', 'OpenPoseXL2.safetensors', cn_dir)
        src_cn = os.path.join(cn_dir, 'OpenPoseXL2.safetensors')
        if os.path.exists(src_cn): os.rename(src_cn, cn_pose)
        import shutil, os as _os
        _os.makedirs(os.path.dirname(drive_cn), exist_ok=True)
        if os.path.exists(cn_pose): shutil.copy2(cn_pose, drive_cn); print('  Cached on Drive')
else:
    print('  Cache: controlnet-openpose-sdxl.safetensors')

print('\nImage models ready! (v3.2: +ControlNet +DWPose +Anime)')

## 6. Audio-Modelle herunterladen

In [ ]:
!pip install transformers accelerate librosa soundfile kokoro -q

print('=== Whisper Large V3 ===')
snapshot_download('openai/whisper-large-v3', cache_dir=CACHE, ignore_patterns=['*.md', '*.txt'])
print('OK')

print('\n=== Qwen2-Audio ===')
snapshot_download('Qwen/Qwen2-Audio-7B-Instruct', cache_dir=CACHE, ignore_patterns=['*.md', '*.txt'])
print('OK')

print('\n=== Kokoro TTS (82M) ===')
snapshot_download('hexgrad/Kokoro-82M', cache_dir=CACHE, ignore_patterns=['*.md'])
print('OK')

print('\nAudio models ready!')

## 6b. RVC Voice Cloning + LivePortrait Face Animation
Optional: Installiert RVC v2 (Voice Cloning) und LivePortrait (Face Animation).
- **RVC**: Eigene Stimmen trainieren aus ~10 Min Audio
- **LivePortrait**: Gesichtsanimation von Video auf Bild uebertragen

In [ ]:
import subprocess, os

print('=' * 60)
print('Installing RVC v2 (Voice Cloning)...')
print('=' * 60)

# RVC dependencies
!pip install -q faiss-cpu pyworld praat-parselmouth torchcrepe soundfile

# Clone RVC
if not os.path.exists('/content/rvc'):
    !git clone --depth=1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI /content/rvc

# Download pretrained models
os.makedirs('/content/rvc/assets/hubert', exist_ok=True)
os.makedirs('/content/rvc/assets/rmvpe', exist_ok=True)
os.makedirs('/content/rvc/assets/pretrained_v2', exist_ok=True)

rvc_files = [
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt',
     '/content/rvc/assets/hubert/hubert_base.pt'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt',
     '/content/rvc/assets/rmvpe/rmvpe.pt'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth',
     '/content/rvc/assets/pretrained_v2/f0G40k.pth'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth',
     '/content/rvc/assets/pretrained_v2/f0D40k.pth'),
]

for url, dest in rvc_files:
    if not os.path.exists(dest):
        print(f'  Downloading {os.path.basename(dest)}...')
        !wget -q -O "{dest}" "{url}"
    else:
        print(f'  {os.path.basename(dest)} exists')

print('\nRVC v2 installed!')

print('\n' + '=' * 60)
print('Installing LivePortrait (Face Animation)...')
print('=' * 60)

!pip install -q onnxruntime-gpu mediapipe insightface tyro dill

if not os.path.exists('/content/LivePortrait'):
    !git clone --depth=1 https://github.com/KwaiVGI/LivePortrait /content/LivePortrait
    !cd /content/LivePortrait && pip install -q -r requirements.txt

# Download weights
print('  Downloading LivePortrait weights...')
!cd /content/LivePortrait && python -c "from src.utils.download import download_models; download_models()" 2>/dev/null || echo 'Weights download — may need manual step'

print('\nLivePortrait installed!')
print('\n' + '=' * 60)
print('DONE! RVC + LivePortrait ready.')
print('=' * 60)


## 7. ComfyUI starten

In [ ]:
import subprocess, threading, time, os

# Alten ComfyUI killen falls er haengt
subprocess.run(['pkill', '-9', '-f', 'main.py.*8188'], capture_output=True)
time.sleep(2)
print('ComfyUI wird gestartet...')

comfy_log = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188',
     '--force-fp16'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd='/content/ComfyUI'
)

# Lese stdout bis ready, dann weiter im Hintergrund
ready = False
for line in iter(comfy_proc.stdout.readline, b''):
    l = line.decode('utf-8', errors='ignore').strip()
    comfy_log.write(l + '\n')
    comfy_log.flush()
    if l: print(l)
    if 'To see the GUI' in l or 'Starting server' in l:
        ready = True
        print()
        print('=== ComfyUI ready! ===')
        break

# Hintergrund-Thread liest restlichen Output (verhindert Buffer-Block)
def drain_stdout():
    for line in iter(comfy_proc.stdout.readline, b''):
        comfy_log.write(line.decode('utf-8', errors='ignore'))
        comfy_log.flush()

threading.Thread(target=drain_stdout, daemon=True).start()
if not ready: print('WARNUNG: ComfyUI hat sich nicht gemeldet!')

## 8. API Server schreiben

In [ ]:
%%writefile /content/api_server.py
"""Cillian AI Studio — API Server v3 (Upscaling + WAN 2.2 + RIFE)"""
import asyncio, json, uuid, os, time, io, tempfile, threading, shutil
from typing import List, Optional
from fastapi import FastAPI, File, UploadFile, Form, HTTPException, Query
from fastapi.responses import FileResponse, JSONResponse
import httpx, uvicorn, torch

app = FastAPI(title='Cillian AI Studio v3')
COMFY = 'http://127.0.0.1:8188'
OUTPUT = '/content/ComfyUI/output'
INPUT = '/content/ComfyUI/input'
CACHE = '/content/drive/MyDrive/ai-models'
KEY = 'ironclaw-video-2026'
os.makedirs(INPUT, exist_ok=True)
os.makedirs(OUTPUT, exist_ok=True)

# Job Store
jobs = {}  # job_id -> {status, result, error, type, started}

def ck(k):
    if k != KEY: raise HTTPException(401, 'Bad API key')

# ════════════════════════════════════════════
# Audio Model Manager
# ════════════════════════════════════════════
audio_loaded = None
audio_models = {}

def unload_audio():
    global audio_loaded, audio_models
    if audio_loaded:
        del audio_models[audio_loaded]
        audio_models = {}
        audio_loaded = None
        torch.cuda.empty_cache()

def load_whisper():
    global audio_loaded, audio_models
    if audio_loaded == 'whisper': return audio_models['whisper']
    unload_audio()
    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        'openai/whisper-large-v3', torch_dtype=torch.float16,
        device_map='auto', cache_dir=CACHE
    )
    processor = AutoProcessor.from_pretrained('openai/whisper-large-v3', cache_dir=CACHE)
    pipe = pipeline('automatic-speech-recognition', model=model, tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor, torch_dtype=torch.float16, device_map='auto')
    audio_models['whisper'] = pipe
    audio_loaded = 'whisper'
    return pipe

def load_qwen():
    global audio_loaded, audio_models
    if audio_loaded == 'qwen': return audio_models['qwen']
    unload_audio()
    from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
    proc = AutoProcessor.from_pretrained('Qwen/Qwen2-Audio-7B-Instruct', cache_dir=CACHE)
    model = Qwen2AudioForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2-Audio-7B-Instruct', torch_dtype=torch.float16,
        device_map='auto', cache_dir=CACHE
    )
    audio_models['qwen'] = (proc, model)
    audio_loaded = 'qwen'
    return proc, model

def load_kokoro():
    global audio_loaded, audio_models
    if audio_loaded == 'kokoro': return audio_models['kokoro']
    unload_audio()
    try:
        from kokoro import KPipeline
        pipe = KPipeline(lang_code='a')
        audio_models['kokoro'] = pipe
        audio_loaded = 'kokoro'
        return pipe
    except Exception as e:
        print(f'Kokoro load error: {e}')
        return None

# ════════════════════════════════════════════
# ComfyUI Workflows — LTX 2.3
# ════════════════════════════════════════════
def ltx_t2v(prompt, w=512, h=512, steps=8, frames=33, cfg=1.0):
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'ltx-2.3-22b-distilled-Q4_K_S.gguf'}},
        '2': {'class_type': 'EmptyLTXVLatentVideo', 'inputs': {'width': w, 'height': h, 'length': frames, 'batch_size': 1}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'ltxv'}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10', 0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10', 0]}},
        '11': {'class_type': 'LTXVConditioning', 'inputs': {'positive': ['3',0], 'negative': ['4',0], 'frame_rate': 24}},
        '12': {'class_type': 'LTXVScheduler', 'inputs': {'steps': steps, 'max_shift': 2.05, 'base_shift': 0.95, 'stretch': True, 'terminal': 0.1}},
        '13': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler'}},
        '14': {'class_type': 'RandomNoise', 'inputs': {'noise_seed': int(time.time()*1000)%2**32}},
        '15': {'class_type': 'BasicGuider', 'inputs': {'model': ['1',0], 'conditioning': ['11',0]}},
        '5': {'class_type': 'SamplerCustomAdvanced', 'inputs': {'guider': ['15',0], 'sampler': ['13',0], 'sigmas': ['12',0], 'noise': ['14',0], 'latent_image': ['2',0]}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'LTX23_video_vae_bf16.safetensors'}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '8': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'ltx_t2v'}},
    }

def ltx_i2v(prompt, image_name, w=512, h=512, steps=8, frames=33, cfg=1.0):
    """LTX 2.3 I2V — image as init latent"""
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'ltx-2.3-22b-distilled-Q4_K_S.gguf'}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'LTX23_video_vae_bf16.safetensors'}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'ltxv'}},
        '2': {'class_type': 'LoadImage', 'inputs': {'image': image_name}},
        '20': {'class_type': 'ImageScale', 'inputs': {'image': ['2',0], 'width': w, 'height': h, 'upscale_method': 'lanczos', 'crop': 'center'}},
        '21': {'class_type': 'VAEEncode', 'inputs': {'pixels': ['20',0], 'vae': ['9',0]}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10',0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['3',0], 'negative': ['4',0], 'latent_image': ['21',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 0.75}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '7': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'ltx_i2v'}},
    }

def wan_t2v(prompt, w=832, h=480, steps=20, frames=81, cfg=5.0):
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'wan2.1-t2v-14b-Q4_K_S.gguf'}},
        '2': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10',0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['3',0], 'negative': ['4',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '8': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'wan_t2v'}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'wan_2.1_vae.safetensors'}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'wan'}}
    }

def wan_i2v(prompt, image_name, w=832, h=480, steps=20, frames=81, cfg=5.0):
    """WAN 2.1 I2V — image as init latent"""
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'wan2.1-i2v-14b-480p-Q4_K_S.gguf'}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'wan_2.1_vae.safetensors'}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'wan'}},
        '2': {'class_type': 'LoadImage', 'inputs': {'image': image_name}},
        '20': {'class_type': 'ImageScale', 'inputs': {'image': ['2',0], 'width': w, 'height': h, 'upscale_method': 'lanczos', 'crop': 'center'}},
        '21': {'class_type': 'VAEEncode', 'inputs': {'pixels': ['20',0], 'vae': ['9',0]}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10',0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['3',0], 'negative': ['4',0], 'latent_image': ['21',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 0.75}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '7': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'wan_i2v'}},
    }

def wan22_t2v(prompt, w=832, h=480, steps=20, frames=81, cfg=5.0):
    """WAN 2.2 MoE T2V — HighNoise + LowNoise GGUF"""
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'Wan2.2-T2V-A14B-HighNoise-Q4_K_S.gguf'}},
        '2': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10',0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['3',0], 'negative': ['4',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '8': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'wan22_t2v'}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'wan_2.1_vae.safetensors'}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'wan'}}
    }

def wan22_i2v(prompt, image_name, w=832, h=480, steps=20, frames=81, cfg=5.0):
    """WAN 2.2 MoE I2V — based on T2V workflow, image as init latent"""
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'Wan2.2-T2V-A14B-HighNoise-Q4_K_S.gguf'}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'wan_2.1_vae.safetensors'}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'wan'}},
        '2': {'class_type': 'LoadImage', 'inputs': {'image': image_name}},
        '20': {'class_type': 'ImageScale', 'inputs': {'image': ['2',0], 'width': w, 'height': h, 'upscale_method': 'lanczos', 'crop': 'center'}},
        '21': {'class_type': 'VAEEncode', 'inputs': {'pixels': ['20',0], 'vae': ['9',0]}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10',0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['3',0], 'negative': ['4',0], 'latent_image': ['21',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 0.75}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '7': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'wan22_i2v'}},
    }

def wan22_ti2v(prompt, image_name, w=832, h=480, steps=20, frames=81, cfg=5.0):
    """WAN 2.2 5B TI2V — image as init latent"""
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'Wan2.2-TI2V-5B-Q4_K_S.gguf'}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'wan_2.1_vae.safetensors'}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'wan'}},
        '2': {'class_type': 'LoadImage', 'inputs': {'image': image_name}},
        '20': {'class_type': 'ImageScale', 'inputs': {'image': ['2',0], 'width': w, 'height': h, 'upscale_method': 'lanczos', 'crop': 'center'}},
        '21': {'class_type': 'VAEEncode', 'inputs': {'pixels': ['20',0], 'vae': ['9',0]}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10',0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['3',0], 'negative': ['4',0], 'latent_image': ['21',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 0.75}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '7': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'wan22_5b_i2v'}},
    }

def flux_t2i(prompt, w=1024, h=1024, steps=20, unet='flux1-dev-Q4_K_S.gguf', cfg=1.0):
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': unet}},
        '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '3': {'class_type': 'DualCLIPLoader', 'inputs': {'clip_name1': 'clip_l.safetensors', 'clip_name2': 't5xxl_fp8_e4m3fn.safetensors', 'type': 'flux'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['3',0]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['3',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler', 'scheduler': 'simple', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['8',0]}},
        '8': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ae.safetensors'}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'flux'}}
    }

def sdxl_t2i(prompt, w=1024, h=1024, steps=4, cfg=1.0):
    return {
        '1': {'class_type': 'UNETLoader', 'inputs': {'unet_name': 'sdxl_lightning_4step_unet.safetensors', 'weight_dtype': 'fp16'}},
        '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '3': {'class_type': 'DualCLIPLoader', 'inputs': {'clip_name1': 'clip_l.safetensors', 'clip_name2': 'clip_g.safetensors', 'type': 'sdxl'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['3',0]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['3',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler', 'scheduler': 'sgm_uniform', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['8',0]}},
        '8': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ae.safetensors'}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'sdxl'}}
    }

def sd35_t2i(prompt, w=1024, h=1024, steps=20, cfg=4.5):
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'sd3.5_medium-Q4_K_S.gguf'}},
        '2': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '3': {'class_type': 'DualCLIPLoader', 'inputs': {'clip_name1': 'clip_l.safetensors', 'clip_name2': 'clip_g.safetensors', 'type': 'sd3'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['3',0]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['3',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler', 'scheduler': 'simple', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['8',0]}},
        '8': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ae.safetensors'}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'sd35'}}
    }

def fluxedup_t2i(prompt, w=1024, h=1024, steps=20, cfg=1.0):
    return flux_t2i(prompt, w, h, steps, 'fluxedupV71_q4ks.gguf', cfg)

def realvis_t2i(prompt, w=1024, h=1024, steps=6, cfg=2.0):
    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'realvisxlV50_lightning.safetensors'}},
        '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['1',1]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'ugly, blurry, deformed, low quality, watermark', 'clip': ['1',1]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'dpmpp_sde', 'scheduler': 'karras', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['1',2]}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'realvis'}}
    }

def juggernaut_t2i(prompt, w=1024, h=1024, steps=25, cfg=6.0):
    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'juggernautXL_ragnarok.safetensors'}},
        '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['1',1]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'ugly, blurry, deformed, low quality', 'clip': ['1',1]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'dpmpp_2m', 'scheduler': 'karras', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['1',2]}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'juggernaut'}}
    }

# ════════════════════════════════════════════
# Upscaling Workflow (NEU v3)
# ════════════════════════════════════════════


# ════════════════════════════════════════════
# Anime Workflows (NEU v3.1)
# ════════════════════════════════════════════
def pony_t2i(prompt, w=1024, h=1024, steps=25, cfg=7.0):
    """Pony Diffusion V6 XL — Anime/Hentai NSFW, score_9 tags"""
    # Auto-prepend quality tags if not present
    if 'score_9' not in prompt:
        prompt = 'score_9, score_8_up, score_7_up, ' + prompt
    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'ponyDiffusionV6XL.safetensors'}},
        '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['1',1]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'score_4, score_3, score_2, score_1, ugly, deformed, bad anatomy, bad hands, missing fingers, extra digits, fewer digits, blurry, lowres, watermark, text, censored', 'clip': ['1',1]}},
        '20': {'class_type': 'CLIPSetLastLayer', 'inputs': {'clip': ['1',1], 'stop_at_clip_layer': -2}},
        '4b': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['20',0]}},
        '7b': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'score_4, score_3, score_2, score_1, ugly, deformed, bad anatomy, bad hands, missing fingers, extra digits, fewer digits, blurry, lowres, watermark, text, censored', 'clip': ['20',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4b',0], 'negative': ['7b',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler_ancestral', 'scheduler': 'normal', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['1',2]}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'pony'}}
    }

def noobai_t2i(prompt, w=1024, h=1024, steps=28, cfg=5.0):
    """NoobAI XL v1.1 Epsilon — Anime NSFW, Danbooru tags"""
    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'NoobAI-XL-v1.1.safetensors'}},
        '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '20': {'class_type': 'CLIPSetLastLayer', 'inputs': {'clip': ['1',1], 'stop_at_clip_layer': -2}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['20',0]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'low quality, worst quality, normal quality, text, signature, jpeg artifacts, bad anatomy, old, early, watermark, artist name, blurry', 'clip': ['20',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'dpmpp_2m', 'scheduler': 'karras', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['1',2]}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'noobai'}}
    }

def animagine_t2i(prompt, w=1024, h=1024, steps=28, cfg=5.0):
    """Animagine XL 4.0 — Clean anime style"""
    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'animagine-xl-4.0.safetensors'}},
        '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['1',1]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'lowres, bad anatomy, bad hands, text, error, missing finger, extra digits, fewer digits, worst quality, low quality, signature, watermark, blurry', 'clip': ['1',1]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler_ancestral', 'scheduler': 'normal', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['1',2]}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'animagine'}}
    }

def upscale_image_wf(image_name, target_w=1920, target_h=1080):
    """4x-UltraSharp → resize to target"""
    return {
        '1': {'class_type': 'LoadImage', 'inputs': {'image': image_name}},
        '2': {'class_type': 'UpscaleModelLoader', 'inputs': {'model_name': '4x-UltraSharp.pth'}},
        '3': {'class_type': 'ImageUpscaleWithModel', 'inputs': {'upscale_model': ['2',0], 'image': ['1',0]}},
        '4': {'class_type': 'ImageScale', 'inputs': {'image': ['3',0], 'width': target_w, 'height': target_h, 'upscale_method': 'lanczos', 'crop': 'center'}},
        '5': {'class_type': 'SaveImage', 'inputs': {'images': ['4',0], 'filename_prefix': 'upscaled'}},
    }

# ════════════════════════════════════════════
# ComfyUI Runner
# ════════════════════════════════════════════
async def run_wf(wf):
    unload_audio()
    cid = str(uuid.uuid4())
    import glob as _glob
    existing = set()
    for d in ['/content/ComfyUI/output', '/content/ComfyUI/temp']:
        existing.update(_glob.glob(os.path.join(d, '**', '*'), recursive=True))
    async with httpx.AsyncClient(timeout=600) as c:
        r = await c.post(f'{COMFY}/prompt', json={'prompt': wf, 'client_id': cid})
        if r.status_code != 200: return None, f'ComfyUI error {r.status_code}: {r.text[:200]}'
        pid = r.json().get('prompt_id')
        if not pid: return None, f'No prompt_id: {r.text[:200]}'
        for attempt in range(300):
            await asyncio.sleep(2)
            h = await c.get(f'{COMFY}/history/{pid}')
            hist = h.json()
            if pid in hist:
                outputs = hist[pid].get('outputs', {})
                for nid, out in outputs.items():
                    for k in ['gifs', 'images', 'videos']:
                        for fi in out.get(k, []):
                            fn = fi.get('filename', '')
                            sf = fi.get('subfolder', '')
                            for base in ['/content/ComfyUI/output', '/content/ComfyUI/temp', OUTPUT]:
                                fp = os.path.join(base, sf, fn) if sf else os.path.join(base, fn)
                                if os.path.exists(fp): return fp, None
                for d2 in ['/content/ComfyUI/output', '/content/ComfyUI/temp']:
                    current = set(_glob.glob(os.path.join(d2, '**', '*'), recursive=True))
                    new_files = current - existing
                    media = [f for f in new_files if f.endswith(('.mp4','.webm','.webp','.png','.jpg','.gif'))]
                    if media:
                        media.sort(key=os.path.getmtime, reverse=True)
                        return media[0], None
                return None, 'No output file found'
        return None, 'Timeout (10 min)'

async def upload_image(image: UploadFile):
    name = f'input_{uuid.uuid4().hex[:8]}.png'
    ipath = os.path.join(INPUT, name)
    with open(ipath, 'wb') as f: f.write(await image.read())
    async with httpx.AsyncClient() as c:
        files = {'image': (name, open(ipath, 'rb'), 'image/png')}
        await c.post(f'{COMFY}/upload/image', files=files)
    return name

async def upload_image_from_path(filepath):
    """Upload existing file to ComfyUI input"""
    name = f'up_{uuid.uuid4().hex[:8]}{os.path.splitext(filepath)[1]}'
    ipath = os.path.join(INPUT, name)
    shutil.copy2(filepath, ipath)
    async with httpx.AsyncClient() as c:
        files = {'image': (name, open(ipath, 'rb'), 'image/png')}
        await c.post(f'{COMFY}/upload/image', files=files)
    return name

async def frames_to_video(filepath, fps=24):
    if not filepath.endswith(('.png','.webp','.jpg')):
        return filepath
    import glob as _g, subprocess
    outdir = os.path.dirname(filepath)
    prefix = os.path.basename(filepath).split('_0')[0]
    mp4path = os.path.join(outdir, prefix + '.mp4')
    frames = sorted(_g.glob(os.path.join(outdir, prefix + '*.png')))
    if len(frames) > 1:
        subprocess.run(['ffmpeg','-y','-framerate',str(fps),'-i',os.path.join(outdir,prefix+'_%05d_.png'),'-c:v','libx264','-pix_fmt','yuv420p',mp4path], capture_output=True)
    elif len(frames) == 1:
        subprocess.run(['ffmpeg','-y','-loop','1','-i',filepath,'-c:v','libx264','-t','3','-pix_fmt','yuv420p',mp4path], capture_output=True)
    if os.path.exists(mp4path) and os.path.getsize(mp4path) > 0:
        return mp4path
    return filepath

# ════════════════════════════════════════════
# Upscale Helpers (NEU v3)
# ════════════════════════════════════════════
async def do_upscale_image(filepath, target_w=1920, target_h=1080):
    """Upscale single image via ComfyUI 4x-UltraSharp"""
    img_name = await upload_image_from_path(filepath)
    wf = upscale_image_wf(img_name, target_w, target_h)
    fp, err = await run_wf(wf)
    if err: return filepath  # Fallback: original
    return fp

async def do_upscale_video(video_path, target_w=1920, target_h=1080, do_rife=True):
    """Upscale video: extract frames → upscale → optional RIFE 2x → reassemble"""
    import subprocess, glob as _glob

    work_dir = f'/content/vid_up_{uuid.uuid4().hex[:8]}'
    frames_in = os.path.join(work_dir, 'in')
    frames_up = os.path.join(work_dir, 'up')
    os.makedirs(frames_in, exist_ok=True)
    os.makedirs(frames_up, exist_ok=True)

    try:
        # 1. Extract frames + get fps
        probe = subprocess.run(['ffprobe', '-v', 'quiet', '-print_format', 'json',
            '-show_streams', video_path], capture_output=True, text=True)
        try:
            streams = json.loads(probe.stdout)['streams']
            r_fps = streams[0].get('r_frame_rate', '24/1')
            num, den = map(int, r_fps.split('/'))
            orig_fps = num / den if den else 24
        except:
            orig_fps = 24

        subprocess.run(['ffmpeg', '-y', '-i', video_path,
            os.path.join(frames_in, 'f_%05d.png')], capture_output=True, check=True)

        frames = sorted(_glob.glob(os.path.join(frames_in, 'f_*.png')))
        if not frames:
            return video_path

        # 2. Upscale each frame
        for i, frame in enumerate(frames):
            img_name = await upload_image_from_path(frame)
            wf = upscale_image_wf(img_name, target_w, target_h)
            fp, err = await run_wf(wf)
            if err:
                # Fallback: ffmpeg resize
                out_f = os.path.join(frames_up, f'f_{i:05d}.png')
                subprocess.run(['ffmpeg', '-y', '-i', frame,
                    '-vf', f'scale={target_w}:{target_h}:flags=lanczos',
                    out_f], capture_output=True)
            else:
                shutil.copy2(fp, os.path.join(frames_up, f'f_{i:05d}.png'))

        # 3. Determine output fps
        out_fps = orig_fps
        if do_rife:
            # RIFE 2x via ffmpeg minterpolate (simpler, works everywhere)
            out_fps = min(orig_fps * 2, 60)

        # 4. Reassemble
        output_mp4 = os.path.join(OUTPUT, f'upscaled_{uuid.uuid4().hex[:8]}.mp4')

        if do_rife:
            # First assemble at original fps, then minterpolate
            temp_mp4 = os.path.join(work_dir, 'temp.mp4')
            subprocess.run(['ffmpeg', '-y', '-framerate', str(orig_fps),
                '-i', os.path.join(frames_up, 'f_%05d.png'),
                '-c:v', 'libx264', '-crf', '18', '-pix_fmt', 'yuv420p',
                temp_mp4], capture_output=True, check=True)
            # Interpolate to double fps
            subprocess.run(['ffmpeg', '-y', '-i', temp_mp4,
                '-vf', f'minterpolate=fps={int(out_fps)}:mi_mode=mci:mc_mode=aobmc:vsbmc=1',
                '-c:v', 'libx264', '-crf', '18', '-pix_fmt', 'yuv420p',
                '-movflags', '+faststart', output_mp4], capture_output=True, check=True)
        else:
            subprocess.run(['ffmpeg', '-y', '-framerate', str(orig_fps),
                '-i', os.path.join(frames_up, 'f_%05d.png'),
                '-c:v', 'libx264', '-crf', '18', '-pix_fmt', 'yuv420p',
                '-movflags', '+faststart', output_mp4], capture_output=True, check=True)

        if os.path.exists(output_mp4) and os.path.getsize(output_mp4) > 0:
            return output_mp4
        return video_path
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)

# ════════════════════════════════════════════
# Async Job Runner
# ════════════════════════════════════════════
async def run_job(job_id, coro):
    try:
        result = await coro
        jobs[job_id]['status'] = 'completed'
        jobs[job_id]['result'] = result
    except Exception as e:
        jobs[job_id]['status'] = 'failed'
        jobs[job_id]['error'] = str(e)

def start_job(job_id, job_type, coro):
    jobs[job_id] = {'status': 'processing', 'type': job_type, 'started': time.time()}
    asyncio.get_event_loop().create_task(run_job(job_id, coro))

# ════════════════════════════════════════════
# API Endpoints
# ════════════════════════════════════════════
@app.get('/health')
async def health():
    return {
        'status': 'ok', 'version': 'v3',
        'models': {
            'text-to-video': {'ltx': '~2min', 'wan': '~4min (2.1)', 'wan22': '~4min (2.2 MoE)', 'wan22-5b': '~2min (2.2 5B)'},
            'image-to-video': {'ltx': '~2min', 'wan': '~4min (2.1)', 'wan22': '~4min (2.2 MoE)', 'wan22-5b': '~2min (2.2 5B)'},
            'text-to-image': {'flux': '~80s', 'flux-schnell': '~15s', 'sdxl': '~30s', 'sd35': '~40s', 'fluxedup': '~80s', 'realvis': '~20s', 'juggernaut': '~60s', 'pony': '~30s (Anime NSFW)', 'noobai': '~35s (Anime NSFW)', 'animagine': '~35s (Anime)'},
            'upscale': {'image': '~5s', 'video': '~3s/frame'},
            'pose-transfer': {'image': '~30s', 'video': '~30s/frame'},
            'speech-to-text': {'whisper': '~5s'},
            'audio-ask': {'qwen2-audio': '~10s'},
            'text-to-speech': {'kokoro': '~2s'}
        }
    }

@app.get('/api/status/{job_id}')
async def job_status(job_id: str):
    if job_id not in jobs: raise HTTPException(404, 'Job not found')
    job = jobs[job_id]
    elapsed = int(time.time() - job['started'])
    if job['status'] == 'completed':
        return {'status': 'completed', 'type': job['type'], 'elapsed': elapsed}
    if job['status'] == 'failed':
        return {'status': 'failed', 'error': job.get('error', 'Unknown'), 'elapsed': elapsed}
    return {'status': 'processing', 'type': job['type'], 'elapsed': elapsed}

@app.get('/api/result/{job_id}')
async def job_result(job_id: str, api_key: str = ''):
    ck(api_key)
    if job_id not in jobs: raise HTTPException(404, 'Job not found')
    job = jobs[job_id]
    if job['status'] != 'completed': raise HTTPException(202, 'Not ready yet')
    result = job['result']
    if len(jobs) > 50:
        old = sorted(jobs.items(), key=lambda x: x[1]['started'])[:20]
        for k, _ in old: del jobs[k]
    if isinstance(result, str) and os.path.exists(result):
        ext = os.path.splitext(result)[1].lower()
        mt = {'.mp4': 'video/mp4', '.wav': 'audio/wav', '.png': 'image/png', '.jpg': 'image/jpeg', '.webp': 'image/webp'}.get(ext, 'application/octet-stream')
        return FileResponse(result, media_type=mt)
    if isinstance(result, dict): return JSONResponse(result)
    return JSONResponse({'result': str(result)})

@app.get('/api/clear')
async def clear_queue(api_key: str = ''):
    ck(api_key)
    async with httpx.AsyncClient(timeout=10) as c:
        try:
            q = await c.get(COMFY + '/queue')
            qdata = q.json()
            for item in qdata.get('queue_running', []) + qdata.get('queue_pending', []):
                pid = item[1] if len(item) > 1 else None
                if pid: await c.post(COMFY + '/queue', json={'delete': [pid]})
            await c.post(COMFY + '/interrupt')
        except: pass
    jobs.clear()
    return {'status': 'cleared'}

# ── Video (v3: +wan22, +upscale, +cfg) ──
async def _do_t2v(prompt, model, width, height, steps, frames, cfg, upscale_to):
    wf_map = {
        'ltx': lambda: ltx_t2v(prompt, width, height, steps, frames, cfg),
        'wan': lambda: wan_t2v(prompt, width, height, steps, frames, cfg),
        'wan22': lambda: wan22_t2v(prompt, width, height, steps, frames, cfg),
    }
    if model not in wf_map: raise Exception(f'model must be: {", ".join(wf_map.keys())}')
    fp, err = await run_wf(wf_map[model]())
    if err: raise Exception(err)
    fps = 24 if model == 'ltx' else 16
    fp = await frames_to_video(fp, fps)
    if upscale_to:
        fp = await do_upscale_video(fp, upscale_to[0], upscale_to[1])
    return fp

@app.post('/api/text-to-video')
async def text_to_video(prompt: str = Form(...), model: str = Form('ltx'),
    width: int = Form(768), height: int = Form(512), steps: int = Form(8),
    frames: int = Form(97), cfg: float = Form(5.0),
    upscale: bool = Form(False), upscale_width: int = Form(1920),
    upscale_height: int = Form(1080), api_key: str = Form(...)):
    ck(api_key)
    up = (upscale_width, upscale_height) if upscale else None
    job_id = uuid.uuid4().hex[:16]
    start_job(job_id, 'video', _do_t2v(prompt, model, width, height, steps, frames, cfg, up))
    return {'job_id': job_id, 'status': 'processing', 'type': 'video'}

async def _do_i2v(prompt, img_name, model, width, height, steps, frames, cfg, upscale_to):
    wf_map = {
        'ltx': lambda: ltx_i2v(prompt, img_name, width, height, steps, frames, cfg),
        'wan': lambda: wan_i2v(prompt, img_name, width, height, steps, frames, cfg),
        'wan22': lambda: wan22_i2v(prompt, img_name, width, height, steps, frames, cfg),
        'wan22-5b': lambda: wan22_ti2v(prompt, img_name, width, height, steps, frames, cfg),
    }
    if model not in wf_map: raise Exception(f'model must be: {", ".join(wf_map.keys())}')
    fp, err = await run_wf(wf_map[model]())
    if err: raise Exception(err)
    fps = 24 if model == 'ltx' else 16
    fp = await frames_to_video(fp, fps)
    if upscale_to:
        fp = await do_upscale_video(fp, upscale_to[0], upscale_to[1])
    return fp

@app.post('/api/image-to-video')
async def image_to_video(prompt: str = Form(...), image: UploadFile = File(...),
    model: str = Form('ltx'), width: int = Form(768), height: int = Form(512),
    steps: int = Form(8), frames: int = Form(97), cfg: float = Form(5.0),
    upscale: bool = Form(False), upscale_width: int = Form(1920),
    upscale_height: int = Form(1080), api_key: str = Form(...)):
    ck(api_key)
    img_name = await upload_image(image)
    up = (upscale_width, upscale_height) if upscale else None
    job_id = uuid.uuid4().hex[:16]
    start_job(job_id, 'video', _do_i2v(prompt, img_name, model, width, height, steps, frames, cfg, up))
    return {'job_id': job_id, 'status': 'processing', 'type': 'video'}

# ── Images (v3: +upscale, +cfg) ──
async def _do_t2i(prompt, model, width, height, steps, cfg, upscale_to):
    wf_map = {
        'flux': lambda: flux_t2i(prompt, width, height, steps, 'flux1-dev-Q4_K_S.gguf', cfg),
        'flux-schnell': lambda: flux_t2i(prompt, width, height, min(steps, 4), 'flux1-schnell-Q4_K_S.gguf', cfg),
        'sdxl': lambda: sdxl_t2i(prompt, width, height, min(steps, 4), cfg),
        'sd35': lambda: sd35_t2i(prompt, width, height, steps, cfg),
        'fluxedup': lambda: fluxedup_t2i(prompt, width, height, steps, cfg),
        'realvis': lambda: realvis_t2i(prompt, width, height, steps, cfg),
        'juggernaut': lambda: juggernaut_t2i(prompt, width, height, steps, cfg),
        'pony': lambda: pony_t2i(prompt, width, height, steps, cfg),
        'noobai': lambda: noobai_t2i(prompt, width, height, steps, cfg),
        'animagine': lambda: animagine_t2i(prompt, width, height, steps, cfg),
    }
    if model not in wf_map: raise Exception(f'model must be: {", ".join(wf_map.keys())}')
    fp, err = await run_wf(wf_map[model]())
    if err: raise Exception(err)
    if upscale_to:
        fp = await do_upscale_image(fp, upscale_to[0], upscale_to[1])
    return fp

@app.post('/api/text-to-image')
async def text_to_image(prompt: str = Form(...), model: str = Form('flux'),
    width: int = Form(1024), height: int = Form(1024), steps: int = Form(20),
    cfg: float = Form(1.0), upscale: bool = Form(False),
    upscale_width: int = Form(1920), upscale_height: int = Form(1080),
    api_key: str = Form(...)):
    ck(api_key)
    up = (upscale_width, upscale_height) if upscale else None
    job_id = uuid.uuid4().hex[:16]
    start_job(job_id, 'image', _do_t2i(prompt, model, width, height, steps, cfg, up))
    return {'job_id': job_id, 'status': 'processing', 'type': 'image'}

# ── Standalone Upscale (NEU v3) ──
@app.post('/api/upscale-image')
async def upscale_image_ep(image: UploadFile = File(...),
    width: int = Form(1920), height: int = Form(1080), api_key: str = Form(...)):
    ck(api_key)
    path = os.path.join(INPUT, f'upimg_{uuid.uuid4().hex[:8]}.png')
    with open(path, 'wb') as f: f.write(await image.read())
    job_id = uuid.uuid4().hex[:16]
    start_job(job_id, 'upscale', do_upscale_image(path, width, height))
    return {'job_id': job_id, 'status': 'processing', 'type': 'upscale'}

@app.post('/api/upscale-video')
async def upscale_video_ep(video: UploadFile = File(...),
    width: int = Form(1920), height: int = Form(1080),
    interpolate: bool = Form(True), api_key: str = Form(...)):
    ck(api_key)
    path = os.path.join(INPUT, f'upvid_{uuid.uuid4().hex[:8]}.mp4')
    with open(path, 'wb') as f: f.write(await video.read())
    job_id = uuid.uuid4().hex[:16]
    start_job(job_id, 'video-upscale', do_upscale_video(path, width, height, interpolate))
    return {'job_id': job_id, 'status': 'processing', 'type': 'video-upscale'}

# ── Audio (unchanged) ──
async def _do_stt(audio_path):
    pipe = load_whisper()
    result = pipe(audio_path, generate_kwargs={'language': 'de', 'task': 'transcribe'})
    return {'text': result['text']}

@app.post('/api/speech-to-text')
async def speech_to_text(audio: UploadFile = File(...), api_key: str = Form(...)):
    ck(api_key)
    path = os.path.join(INPUT, f'stt_{uuid.uuid4().hex[:8]}.wav')
    with open(path, 'wb') as f: f.write(await audio.read())
    job_id = uuid.uuid4().hex[:16]
    start_job(job_id, 'stt', _do_stt(path))
    return {'job_id': job_id, 'status': 'processing', 'type': 'stt'}

async def _do_audio_ask(audio_path, question):
    import librosa
    proc, model = load_qwen()
    conversation = [{'role': 'user', 'content': [
        {'type': 'audio', 'audio_url': audio_path},
        {'type': 'text', 'text': question}
    ]}]
    text = proc.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios = [librosa.load(audio_path, sr=proc.feature_extractor.sampling_rate)[0]]
    inputs = proc(text=text, audios=audios, return_tensors='pt', padding=True).to('cuda')
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=512)
    ids = ids[:, inputs.input_ids.size(1):]
    return {'answer': proc.batch_decode(ids, skip_special_tokens=True)[0]}

@app.post('/api/audio-ask')
async def audio_ask(audio: UploadFile = File(...), question: str = Form('Was hoerst du?'),
    api_key: str = Form(...)):
    ck(api_key)
    path = os.path.join(INPUT, f'ask_{uuid.uuid4().hex[:8]}.wav')
    with open(path, 'wb') as f: f.write(await audio.read())
    job_id = uuid.uuid4().hex[:16]
    start_job(job_id, 'audio', _do_audio_ask(path, question))
    return {'job_id': job_id, 'status': 'processing', 'type': 'audio'}

async def _do_tts(text, voice):
    import soundfile as sf, numpy as np
    pipe = load_kokoro()
    if not pipe: raise Exception('Kokoro not available')
    samples = []
    for chunk in pipe(text, voice=voice):
        samples.append(chunk[2])
    audio = np.concatenate(samples)
    path = os.path.join(OUTPUT, f'tts_{uuid.uuid4().hex[:8]}.wav')
    sf.write(path, audio, 24000)
    return path

@app.post('/api/text-to-speech')
async def text_to_speech(text: str = Form(...), voice: str = Form('af_heart'),
    api_key: str = Form(...)):
    ck(api_key)
    job_id = uuid.uuid4().hex[:16]
    start_job(job_id, 'tts', _do_tts(text, voice))
    return {'job_id': job_id, 'status': 'processing', 'type': 'tts'}

# ── Custom Workflow ──
@app.post('/api/custom-workflow')
async def custom_workflow(api_key: str = Form(...), workflow: str = Form(...)):
    ck(api_key)
    import json as _json
    wf = _json.loads(workflow)
    job_id = uuid.uuid4().hex[:16]
    async def _run():
        fp, err = await run_wf(wf)
        if err: raise Exception(err)
        return fp
    start_job(job_id, 'custom', _run())
    return {'job_id': job_id, 'status': 'processing', 'type': 'custom'}



# ════════════════════════════════════════════
# Video-to-Video: Pose Transfer (NEU v3.2)
# ════════════════════════════════════════════
def pose_transfer_wf(pose_image_name, prompt, w=1024, h=1024, steps=25, cfg=7.0, cn_strength=0.85, ckpt='ponyDiffusionV6XL.safetensors'):
    """Extract pose from reference image, generate character in that pose"""
    # Auto-prepend quality tags for Pony
    if 'pony' in ckpt.lower() and 'score_9' not in prompt:
        prompt = 'score_9, score_8_up, score_7_up, ' + prompt
    neg = 'score_4, score_3, score_2, score_1, ugly, deformed, bad anatomy, bad hands, missing fingers, extra digits, blurry, lowres, watermark, censored' if 'pony' in ckpt.lower() else 'ugly, deformed, bad anatomy, blurry, lowres'
    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': ckpt}},
        '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '20': {'class_type': 'CLIPSetLastLayer', 'inputs': {'clip': ['1',1], 'stop_at_clip_layer': -2}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['20',0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': neg, 'clip': ['20',0]}},
        # Load pose reference image
        '30': {'class_type': 'LoadImage', 'inputs': {'image': pose_image_name}},
        # DWPose extraction
        '31': {'class_type': 'DWPreprocessor', 'inputs': {'image': ['30',0], 'detect_hand': 'enable', 'detect_body': 'enable', 'detect_face': 'enable', 'resolution': max(w, h)}},
        # ControlNet
        '32': {'class_type': 'ControlNetLoader', 'inputs': {'control_net_name': 'controlnet-openpose-sdxl.safetensors'}},
        '33': {'class_type': 'ControlNetApplyAdvanced', 'inputs': {'positive': ['3',0], 'negative': ['4',0], 'control_net': ['32',0], 'image': ['31',0], 'strength': cn_strength, 'start_percent': 0.0, 'end_percent': 1.0}},
        # Sample
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['33',0], 'negative': ['33',1], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': cfg, 'sampler_name': 'euler_ancestral', 'scheduler': 'normal', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['1',2]}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'pose_transfer'}},
    }

async def _do_pose_transfer(prompt, pose_image_name, model, width, height, steps, cfg, cn_strength):
    ckpt_map = {
        'pony': 'ponyDiffusionV6XL.safetensors',
        'noobai': 'NoobAI-XL-v1.1.safetensors',
        'animagine': 'animagine-xl-4.0.safetensors',
        'juggernaut': 'juggernautXL_ragnarok.safetensors',
        'realvis': 'realvisxlV50_lightning.safetensors',
    }
    ckpt = ckpt_map.get(model, 'ponyDiffusionV6XL.safetensors')
    wf = pose_transfer_wf(pose_image_name, prompt, width, height, steps, cfg, cn_strength, ckpt)
    fp, err = await run_wf(wf)
    if err: raise Exception(err)
    return fp

@app.post('/api/pose-transfer')
async def pose_transfer(prompt: str = Form(...), image: UploadFile = File(...),
    model: str = Form('pony'), width: int = Form(832), height: int = Form(1216),
    steps: int = Form(25), cfg: float = Form(7.0), cn_strength: float = Form(0.85),
    api_key: str = Form(...)):
    ck(api_key)
    img_name = await upload_image(image)
    job_id = uuid.uuid4().hex[:16]
    start_job(job_id, 'pose', _do_pose_transfer(prompt, img_name, model, width, height, steps, cfg, cn_strength))
    return {'job_id': job_id, 'status': 'processing', 'type': 'pose'}

# Video-to-Video: Frame-by-frame pose transfer
@app.post('/api/video-pose-transfer')
async def video_pose_transfer(prompt: str = Form(...), video: UploadFile = File(...),
    model: str = Form('pony'), width: int = Form(832), height: int = Form(1216),
    steps: int = Form(20), cfg: float = Form(7.0), cn_strength: float = Form(0.8),
    fps: int = Form(12), api_key: str = Form(...)):
    ck(api_key)
    vid_path = os.path.join(INPUT, f'refvid_{uuid.uuid4().hex[:8]}.mp4')
    with open(vid_path, 'wb') as f: f.write(await video.read())
    job_id = uuid.uuid4().hex[:16]

    async def _run():
        import subprocess, glob as _glob
        work_dir = f'/content/v2v_{uuid.uuid4().hex[:8]}'
        frames_in = os.path.join(work_dir, 'in')
        frames_out = os.path.join(work_dir, 'out')
        os.makedirs(frames_in, exist_ok=True)
        os.makedirs(frames_out, exist_ok=True)
        try:
            # Extract frames (limit to fps)
            subprocess.run(['ffmpeg', '-y', '-i', vid_path, '-vf', f'fps={fps}',
                os.path.join(frames_in, 'f_%05d.png')], capture_output=True, check=True)
            frames = sorted(_glob.glob(os.path.join(frames_in, 'f_*.png')))
            if not frames: raise Exception('No frames extracted')
            # Process each frame
            for i, frame in enumerate(frames):
                img_name = await upload_image_from_path(frame)
                fp, err = await run_wf(pose_transfer_wf(img_name, prompt, width, height, steps, cfg, cn_strength,
                    {'pony':'ponyDiffusionV6XL.safetensors','noobai':'NoobAI-XL-v1.1.safetensors','animagine':'animagine-xl-4.0.safetensors'}.get(model,'ponyDiffusionV6XL.safetensors')))
                if err: continue
                shutil.copy2(fp, os.path.join(frames_out, f'f_{i:05d}.png'))
            # Assemble
            output = os.path.join(OUTPUT, f'v2v_{uuid.uuid4().hex[:8]}.mp4')
            subprocess.run(['ffmpeg', '-y', '-framerate', str(fps),
                '-i', os.path.join(frames_out, 'f_%05d.png'),
                '-c:v', 'libx264', '-crf', '18', '-pix_fmt', 'yuv420p',
                '-movflags', '+faststart', output], capture_output=True, check=True)
            return output
        finally:
            shutil.rmtree(work_dir, ignore_errors=True)

    start_job(job_id, 'video-pose', _run())
    return {'job_id': job_id, 'status': 'processing', 'type': 'video-pose'}



# ════════════════════════════════════════════
# Audio-Video Merge (NEU v3.3)
# ════════════════════════════════════════════
@app.post('/api/merge-audio-video')
async def merge_audio_video(video: UploadFile = File(...), audio: UploadFile = File(...),
    api_key: str = Form(...)):
    ck(api_key)
    vid_path = os.path.join(INPUT, f'merge_v_{uuid.uuid4().hex[:8]}.mp4')
    aud_path = os.path.join(INPUT, f'merge_a_{uuid.uuid4().hex[:8]}.wav')
    with open(vid_path, 'wb') as f: f.write(await video.read())
    with open(aud_path, 'wb') as f: f.write(await audio.read())
    job_id = uuid.uuid4().hex[:16]
    async def _run():
        import subprocess
        output = os.path.join(OUTPUT, f'merged_{uuid.uuid4().hex[:8]}.mp4')
        subprocess.run([
            'ffmpeg', '-y', '-i', vid_path, '-i', aud_path,
            '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k',
            '-map', '0:v:0', '-map', '1:a:0', '-shortest',
            '-movflags', '+faststart', output
        ], capture_output=True, check=True)
        return output
    start_job(job_id, 'merge', _run())
    return {'job_id': job_id, 'status': 'processing', 'type': 'merge'}

# Named TTS — Generate speech with specific voice
@app.post('/api/tts-voice')
async def tts_voice(text: str = Form(...), voice: str = Form('af_heart'),
    speed: float = Form(1.0), api_key: str = Form(...)):
    ck(api_key)
    job_id = uuid.uuid4().hex[:16]
    async def _run():
        import soundfile as sf, numpy as np
        pipe = load_kokoro()
        if not pipe: raise Exception('Kokoro not available')
        # Map voice names to Kokoro codes
        voice_map = {
            'af_heart': 'af_heart', 'af_bella': 'af_bella', 'af_nicole': 'af_nicole',
            'af_sarah': 'af_sarah', 'af_sky': 'af_sky', 'am_adam': 'am_adam',
            'am_michael': 'am_michael', 'bf_emma': 'bf_emma', 'bf_isabella': 'bf_isabella',
            'bm_george': 'bm_george', 'bm_lewis': 'bm_lewis',
        }
        v = voice_map.get(voice, voice)
        samples = []
        for chunk in pipe(text, voice=v, speed=speed):
            samples.append(chunk[2])
        audio = np.concatenate(samples)
        path_out = os.path.join(OUTPUT, f'voice_{uuid.uuid4().hex[:8]}.wav')
        sf.write(path_out, audio, 24000)
        return path_out
    start_job(job_id, 'tts', _run())
    return {'job_id': job_id, 'status': 'processing', 'type': 'tts'}

# List available Kokoro voices
@app.get('/api/voices')
async def list_voices():
    return {
        'voices': [
            {'id': 'af_heart', 'name': 'Heart (Female US)', 'lang': 'en', 'gender': 'female'},
            {'id': 'af_bella', 'name': 'Bella (Female US)', 'lang': 'en', 'gender': 'female'},
            {'id': 'af_nicole', 'name': 'Nicole (Female US)', 'lang': 'en', 'gender': 'female'},
            {'id': 'af_sarah', 'name': 'Sarah (Female US)', 'lang': 'en', 'gender': 'female'},
            {'id': 'af_sky', 'name': 'Sky (Female US)', 'lang': 'en', 'gender': 'female'},
            {'id': 'am_adam', 'name': 'Adam (Male US)', 'lang': 'en', 'gender': 'male'},
            {'id': 'am_michael', 'name': 'Michael (Male US)', 'lang': 'en', 'gender': 'male'},
            {'id': 'bf_emma', 'name': 'Emma (Female UK)', 'lang': 'en', 'gender': 'female'},
            {'id': 'bf_isabella', 'name': 'Isabella (Female UK)', 'lang': 'en', 'gender': 'female'},
            {'id': 'bm_george', 'name': 'George (Male UK)', 'lang': 'en', 'gender': 'male'},
            {'id': 'bm_lewis', 'name': 'Lewis (Male UK)', 'lang': 'en', 'gender': 'male'},
        ]
    }



# ════════════════════════════════════════════
# LoRA Training (NEU v3.3)
# ════════════════════════════════════════════
import base64 as _b64

LORA_DIR = '/content/ComfyUI/models/loras'
TRAIN_DIR = '/content/lora_training'
DRIVE_LORA = '/content/drive/MyDrive/ai-models/loras'

@app.post('/api/lora/upload-training-images')
async def lora_upload_images(api_key: str = Form(...), character_name: str = Form('character')):
    """Upload training images as base64 JSON array. Call multiple times to add more."""
    ck(api_key)
    char_dir = os.path.join(TRAIN_DIR, character_name.replace(' ', '_').lower())
    img_dir = os.path.join(char_dir, 'img', f'20_{character_name}')
    os.makedirs(img_dir, exist_ok=True)
    return {'ok': True, 'upload_dir': img_dir, 'character': character_name}

@app.post('/api/lora/add-image')
async def lora_add_image(image: UploadFile = File(...), character_name: str = Form('character'),
    caption: str = Form(''), api_key: str = Form(...)):
    ck(api_key)
    char_dir = os.path.join(TRAIN_DIR, character_name.replace(' ', '_').lower())
    img_dir = os.path.join(char_dir, 'img', f'20_{character_name}')
    os.makedirs(img_dir, exist_ok=True)
    # Save image
    idx = len([f for f in os.listdir(img_dir) if f.endswith(('.png','.jpg','.jpeg'))])
    fname = f'{idx:03d}.png'
    fpath = os.path.join(img_dir, fname)
    with open(fpath, 'wb') as f: f.write(await image.read())
    # Save caption (same name, .txt)
    if caption:
        with open(fpath.rsplit('.', 1)[0] + '.txt', 'w') as f: f.write(caption)
    count = len([f for f in os.listdir(img_dir) if f.endswith(('.png','.jpg','.jpeg'))])
    return {'ok': True, 'filename': fname, 'total_images': count}

@app.post('/api/lora/train')
async def lora_train(character_name: str = Form('character'),
    base_model: str = Form('ponyDiffusionV6XL.safetensors'),
    steps: int = Form(800), rank: int = Form(32), lr: float = Form(1e-4),
    api_key: str = Form(...)):
    ck(api_key)
    char_dir = os.path.join(TRAIN_DIR, character_name.replace(' ', '_').lower())
    img_dir = os.path.join(char_dir, 'img')
    output_dir = os.path.join(char_dir, 'output')
    os.makedirs(output_dir, exist_ok=True)

    # Check images exist
    train_imgs = os.path.join(img_dir, f'20_{character_name}')
    if not os.path.exists(train_imgs):
        return {'error': 'No training images uploaded'}
    img_count = len([f for f in os.listdir(train_imgs) if f.endswith(('.png','.jpg','.jpeg'))])
    if img_count < 5:
        return {'error': f'Need at least 5 images, got {img_count}'}

    job_id = uuid.uuid4().hex[:16]
    lora_name = f'lora_{character_name.replace(" ", "_").lower()}'

    async def _train():
        import subprocess
        model_path = f'/content/ComfyUI/models/checkpoints/{base_model}'
        if not os.path.exists(model_path):
            raise Exception(f'Base model not found: {base_model}')

        cmd = [
            'python', '/content/sd-scripts/sdxl_train_network.py',
            '--pretrained_model_name_or_path', model_path,
            '--train_data_dir', img_dir,
            '--output_dir', output_dir,
            '--output_name', lora_name,
            '--resolution', '1024,1024',
            '--train_batch_size', '1',
            '--max_train_steps', str(steps),
            '--learning_rate', str(lr),
            '--network_module', 'networks.lora',
            '--network_dim', str(rank),
            '--network_alpha', str(rank),
            '--optimizer_type', 'AdamW8bit',
            '--mixed_precision', 'fp16',
            '--save_precision', 'fp16',
            '--cache_latents',
            '--gradient_checkpointing',
            '--max_data_loader_n_workers', '1',
            '--bucket_reso_steps', '64',
            '--min_bucket_reso', '512',
            '--max_bucket_reso', '1536',
            '--xformers',
            '--save_model_as', 'safetensors',
        ]

        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
        if proc.returncode != 0:
            raise Exception(f'Training failed: {proc.stderr[-500:]}')

        # Copy LoRA to ComfyUI models
        lora_file = os.path.join(output_dir, f'{lora_name}.safetensors')
        if not os.path.exists(lora_file):
            raise Exception('LoRA file not created')

        import shutil
        dest = os.path.join(LORA_DIR, f'{lora_name}.safetensors')
        shutil.copy2(lora_file, dest)

        # Cache on Drive
        os.makedirs(DRIVE_LORA, exist_ok=True)
        shutil.copy2(lora_file, os.path.join(DRIVE_LORA, f'{lora_name}.safetensors'))

        return dest

    start_job(job_id, 'lora-train', _train())
    return {'job_id': job_id, 'status': 'training', 'type': 'lora-train',
            'character': character_name, 'images': img_count, 'steps': steps, 'rank': rank}

@app.get('/api/lora/list')
async def lora_list():
    loras = []
    for d in [LORA_DIR, DRIVE_LORA]:
        try:
            for f in os.listdir(d):
                if f.endswith('.safetensors') and f not in [l['filename'] for l in loras]:
                    loras.append({'filename': f, 'name': f.replace('.safetensors',''), 'path': os.path.join(d, f)})
        except: pass
    return {'loras': loras}

# LoRA-enhanced image generation
def apply_lora_to_workflow(wf, lora_name, strength=0.8):
    """Inject LoRA loader into any workflow that has a CheckpointLoaderSimple"""
    # Find the checkpoint node
    ckpt_node = None
    for nid, node in wf.items():
        if node.get('class_type') == 'CheckpointLoaderSimple':
            ckpt_node = nid
            break
    if not ckpt_node:
        return wf  # No checkpoint found, return as-is

    # Add LoRA loader
    lora_id = str(max(int(k) for k in wf.keys()) + 1)
    wf[lora_id] = {
        'class_type': 'LoraLoader',
        'inputs': {
            'lora_name': lora_name + '.safetensors',
            'strength_model': strength,
            'strength_clip': strength,
            'model': [ckpt_node, 0],
            'clip': [ckpt_node, 1],
        }
    }
    # Redirect nodes that used checkpoint model/clip to use LoRA output
    for nid, node in wf.items():
        if nid == lora_id: continue
        inputs = node.get('inputs', {})
        for key, val in inputs.items():
            if isinstance(val, list) and len(val) == 2:
                if val[0] == ckpt_node and val[1] in [0, 1]:
                    if key in ['model', 'positive', 'negative'] and val[1] == 0:
                        inputs[key] = [lora_id, 0]
                    elif key in ['clip', 'text'] and val[1] == 1:
                        inputs[key] = [lora_id, 1]
    return wf

@app.post('/api/generate-with-lora')
async def generate_with_lora(prompt: str = Form(...), lora_name: str = Form(...),
    model: str = Form('pony'), width: int = Form(832), height: int = Form(1216),
    steps: int = Form(25), cfg: float = Form(7.0), lora_strength: float = Form(0.8),
    api_key: str = Form(...)):
    ck(api_key)
    # Build base workflow
    wf_map = {
        'pony': lambda: pony_t2i(prompt, width, height, steps, cfg),
        'noobai': lambda: noobai_t2i(prompt, width, height, steps, cfg),
        'animagine': lambda: animagine_t2i(prompt, width, height, steps, cfg),
    }
    if model not in wf_map: model = 'pony'
    wf = wf_map[model]()
    # Apply LoRA
    wf = apply_lora_to_workflow(wf, lora_name, lora_strength)
    job_id = uuid.uuid4().hex[:16]
    async def _run():
        fp, err = await run_wf(wf)
        if err: raise Exception(err)
        return fp
    start_job(job_id, 'image-lora', _run())
    return {'job_id': job_id, 'status': 'processing', 'type': 'image-lora'}




# ════════════════════════════════════════════════════════════════
# RVC Voice Cloning Endpoints
# ════════════════════════════════════════════════════════════════
import glob as globmod

RVC_MODELS_DIR = '/content/rvc_models'
RVC_TRAINING_DIR = '/content/rvc_training'
os.makedirs(RVC_MODELS_DIR, exist_ok=True)

@app.get('/api/rvc-models')
async def rvc_models(api_key: str = Query(...)):
    ck(api_key)
    models = []
    if os.path.exists(RVC_MODELS_DIR):
        for d in os.listdir(RVC_MODELS_DIR):
            pth = os.path.join(RVC_MODELS_DIR, d, f'{d}.pth')
            if os.path.exists(pth):
                models.append({'character_id': d, 'size_mb': round(os.path.getsize(pth)/1e6, 1)})
    return {'models': models}

@app.post('/api/rvc-train')
async def rvc_train(character_id: str = Form(...), api_key: str = Form(...), files: List[UploadFile] = File(...)):
    ck(api_key)
    train_dir = os.path.join(RVC_TRAINING_DIR, character_id)
    os.makedirs(train_dir, exist_ok=True)
    for f in files:
        data = await f.read()
        with open(os.path.join(train_dir, f.filename or f'sample.wav'), 'wb') as out:
            out.write(data)
    job_id = uuid.uuid4().hex[:16]
    async def _train():
        import subprocess as sp
        # Merge audio
        audio_files = []
        for ext in ['*.wav','*.mp3','*.m4a','*.ogg']:
            audio_files.extend(globmod.glob(os.path.join(train_dir, ext)))
        merged = os.path.join(train_dir, 'merged.wav')
        if len(audio_files) == 1:
            sp.run(['ffmpeg','-y','-i',audio_files[0],'-ac','1','-ar','40000',merged], capture_output=True, timeout=120)
        else:
            lst = os.path.join(train_dir, 'list.txt')
            with open(lst,'w') as lf:
                for af in audio_files: lf.write(f"file '{af}'\n")
            sp.run(['ffmpeg','-y','-f','concat','-safe','0','-i',lst,'-ac','1','-ar','40000',merged], capture_output=True, timeout=120)
        if not os.path.exists(merged): raise Exception('Audio merge failed')
        model_dir = os.path.join(RVC_MODELS_DIR, character_id)
        os.makedirs(model_dir, exist_ok=True)
        # RVC training pipeline
        sp.run(['python','/content/rvc/trainset_preprocess_pipeline_print.py',merged,'40000','1',model_dir,'True','3.7'], capture_output=True, timeout=600, cwd='/content/rvc')
        sp.run(['python','/content/rvc/extract_f0_print.py',model_dir,'1','rmvpe'], capture_output=True, timeout=300, cwd='/content/rvc')
        sp.run(['python','/content/rvc/extract_feature_print.py','cuda:0','1','0','0',model_dir,'7.0'], capture_output=True, timeout=300, cwd='/content/rvc')
        sp.run(['python','/content/rvc/train_nsf_sim_cache_sid_load_pretrain.py','-e',character_id,'-sr','40000','-f0','1','-bs','8','-g','0','-te','300','-se','50','-pg','/content/rvc/assets/pretrained_v2/f0G40k.pth','-pd','/content/rvc/assets/pretrained_v2/f0D40k.pth','-l','1','-c','0'], capture_output=True, timeout=1200, cwd='/content/rvc')
        model_file = os.path.join(model_dir, f'{character_id}.pth')
        if not os.path.exists(model_file):
            rvc_w = globmod.glob(f'/content/rvc/weights/{character_id}*.pth')
            if rvc_w: shutil.copy(rvc_w[-1], model_file)
        if not os.path.exists(model_file): raise Exception('Model not found after training')
        return model_file
    start_job(job_id, 'rvc-train', _train())
    return {'job_id': job_id, 'status': 'processing', 'type': 'rvc-train'}

@app.post('/api/rvc-convert')
async def rvc_convert(character_id: str = Form(...), api_key: str = Form(...), audio: UploadFile = File(...)):
    ck(api_key)
    model_path = os.path.join(RVC_MODELS_DIR, character_id, f'{character_id}.pth')
    if not os.path.exists(model_path): raise HTTPException(404, f'No RVC model for {character_id}')
    input_path = f'/tmp/rvc_in_{uuid.uuid4().hex[:8]}.wav'
    with open(input_path, 'wb') as f: f.write(await audio.read())
    job_id = uuid.uuid4().hex[:16]
    async def _convert():
        import subprocess as sp
        out = f'/tmp/rvc_out_{character_id}_{uuid.uuid4().hex[:8]}.wav'
        sp.run(['python','/content/rvc/tools/infer_cli.py','0',input_path,out,model_path,'','rmvpe','','0.75','3','0','1','0.33'], capture_output=True, timeout=120, cwd='/content/rvc')
        if os.path.exists(out): return out
        raise Exception('RVC conversion failed')
    start_job(job_id, 'rvc-convert', _convert())
    return {'job_id': job_id, 'status': 'processing', 'type': 'rvc-convert'}

# ════════════════════════════════════════════════════════════════
# LivePortrait Face Animation Endpoints
# ════════════════════════════════════════════════════════════════

@app.post('/api/face-animate')
async def face_animate(api_key: str = Form(...), source_image: UploadFile = File(...), driving_video: UploadFile = File(...), paste_back: str = Form('true')):
    ck(api_key)
    src = f'/tmp/lp_src_{uuid.uuid4().hex[:8]}.png'
    drv = f'/tmp/lp_drv_{uuid.uuid4().hex[:8]}.mp4'
    with open(src, 'wb') as f: f.write(await source_image.read())
    with open(drv, 'wb') as f: f.write(await driving_video.read())
    job_id = uuid.uuid4().hex[:16]
    async def _anim():
        import subprocess as sp
        out_dir = f'/tmp/lp_out_{uuid.uuid4().hex[:8]}'
        os.makedirs(out_dir, exist_ok=True)
        cmd = ['python','/content/LivePortrait/inference.py','-s',src,'-d',drv,'-o',out_dir,'--flag_crop_driving_video']
        if paste_back == 'true': cmd.append('--flag_pasteback')
        sp.run(cmd, capture_output=True, timeout=600, cwd='/content/LivePortrait')
        outputs = globmod.glob(os.path.join(out_dir, '**', '*.mp4'), recursive=True)
        if not outputs: raise Exception('LivePortrait produced no output')
        return outputs[0]
    start_job(job_id, 'face-animate', _anim())
    return {'job_id': job_id, 'status': 'processing', 'type': 'face-animate'}

@app.post('/api/face-body-animate')
async def face_body_animate(api_key: str = Form(...), character_image: UploadFile = File(...), driving_video: UploadFile = File(...), prompt: str = Form(''), model: str = Form('pony'), cn_strength: float = Form(0.85), face_strength: float = Form(1.0), width: int = Form(832), height: int = Form(1216)):
    ck(api_key)
    char_p = f'/tmp/fb_char_{uuid.uuid4().hex[:8]}.png'
    drv_p = f'/tmp/fb_drv_{uuid.uuid4().hex[:8]}.mp4'
    with open(char_p, 'wb') as f: f.write(await character_image.read())
    with open(drv_p, 'wb') as f: f.write(await driving_video.read())
    job_id = uuid.uuid4().hex[:16]
    async def _fb():
        import subprocess as sp
        work = f'/tmp/fb_{uuid.uuid4().hex[:8]}'
        os.makedirs(work, exist_ok=True)
        frames_dir = os.path.join(work, 'frames')
        os.makedirs(frames_dir, exist_ok=True)
        # Extract frames at 12fps
        sp.run(['ffmpeg','-i',drv_p,'-vf',f'fps=12,scale={width}:{height}:force_original_aspect_ratio=decrease,pad={width}:{height}:(ow-iw)/2:(oh-ih)/2',os.path.join(frames_dir,'f_%04d.png')], capture_output=True, timeout=120)
        frames = sorted(globmod.glob(os.path.join(frames_dir, 'f_*.png')))
        if not frames: raise Exception('No frames extracted')
        # Generate body for keyframes via ControlNet
        body_dir = os.path.join(work, 'body')
        os.makedirs(body_dir, exist_ok=True)
        kf_int = max(1, len(frames) // 12)
        last_body = None
        for i, fp in enumerate(frames):
            if i % kf_int == 0:
                try:
                    wf = pose_transfer_workflow(char_p, fp, prompt, model, cn_strength, width, height)
                    result, err = await run_wf(wf)
                    if not err:
                        shutil.copy(result, os.path.join(body_dir, f'b_{i:04d}.png'))
                        last_body = os.path.join(body_dir, f'b_{i:04d}.png')
                        continue
                except: pass
                shutil.copy(fp, os.path.join(body_dir, f'b_{i:04d}.png'))
                last_body = os.path.join(body_dir, f'b_{i:04d}.png')
            else:
                src_f = last_body or fp
                shutil.copy(src_f, os.path.join(body_dir, f'b_{i:04d}.png'))
        # Assemble body video
        body_vid = os.path.join(work, 'body.mp4')
        sp.run(['ffmpeg','-y','-framerate','12','-i',os.path.join(body_dir,'b_%04d.png'),'-c:v','libx264','-crf','18','-pix_fmt','yuv420p',body_vid], capture_output=True, timeout=120)
        if not os.path.exists(body_vid): raise Exception('Body video assembly failed')
        # LivePortrait face animation
        lp_dir = os.path.join(work, 'lp')
        os.makedirs(lp_dir, exist_ok=True)
        sp.run(['python','/content/LivePortrait/inference.py','-s',char_p,'-d',drv_p,'-o',lp_dir,'--flag_pasteback','--flag_crop_driving_video'], capture_output=True, timeout=600, cwd='/content/LivePortrait')
        lp_out = globmod.glob(os.path.join(lp_dir, '**', '*.mp4'), recursive=True)
        if lp_out:
            final = os.path.join(work, 'final.mp4')
            sp.run(['ffmpeg','-y','-i',body_vid,'-i',lp_out[0],'-filter_complex','[1:v]scale=iw:ih[face];[0:v][face]overlay=0:0:shortest=1','-c:v','libx264','-crf','18','-pix_fmt','yuv420p',final], capture_output=True, timeout=120)
            if os.path.exists(final): return final
        return body_vid
    start_job(job_id, 'face-body-animate', _fb())
    return {'job_id': job_id, 'status': 'processing', 'type': 'face-body-animate'}

@app.post('/api/analyze-video-frame')
async def analyze_video_frame(api_key: str = Form(...), video: UploadFile = File(...), question: str = Form('Describe what the person is wearing, holding, and their accessories. Be specific about colors and materials.')):
    ck(api_key)
    vid_path = f'/tmp/vfa_{uuid.uuid4().hex[:8]}.mp4'
    with open(vid_path, 'wb') as f: f.write(await video.read())
    frame_path = f'/tmp/vfa_frame_{uuid.uuid4().hex[:8]}.png'
    import subprocess as sp
    sp.run(['ffmpeg','-y','-i',vid_path,'-vf','select=eq(n\\,5)','-frames:v','1',frame_path], capture_output=True, timeout=30)
    if not os.path.exists(frame_path): raise HTTPException(500, 'Frame extraction failed')
    job_id = uuid.uuid4().hex[:16]
    start_job(job_id, 'video-frame-analyze', _do_audio_ask(frame_path, question))
    return {'job_id': job_id, 'status': 'processing', 'type': 'video-frame-analyze'}




# ════════════════════════════════════════════════════════════════
# IMG2IMG — Image-to-Image Generation (all SDXL models)
# ════════════════════════════════════════════════════════════════

MODEL_CKPTS = {
    'pony': 'ponyDiffusionV6XL.safetensors',
    'noobai': 'noobaiXLNAIXL_vPred10.safetensors',
    'animagine': 'animagine-xl-4.0.safetensors',
    'juggernaut': 'juggernautXL_ragnarok.safetensors',
    'realvis': 'RealVisXL_V5.safetensors',
    'fluxedup': 'fluxedUpV7.1.safetensors',
}

def img2img_wf(image_name, prompt, model='pony', w=832, h=1216, steps=25, cfg=7.0, strength=0.55, neg_prompt=''):
    """Generic img2img workflow for any SDXL model"""
    ckpt = MODEL_CKPTS.get(model, MODEL_CKPTS['pony'])
    is_pony = 'pony' in ckpt.lower()

    if is_pony and 'score_9' not in prompt:
        prompt = 'score_9, score_8_up, score_7_up, score_6_up, rating_explicit, source_anime, ' + prompt
    if not neg_prompt:
        neg_prompt = 'score_4, score_3, score_2, score_1, rating_safe, rating_questionable, censored, bar censor, mosaic censoring, convenient censoring, ugly, deformed, bad anatomy, blurry, lowres, watermark, text, child, loli' if is_pony else 'censored, bar_censor, mosaic_censoring, convenient_censoring, ugly, deformed, bad anatomy, blurry, lowres, watermark, child, loli'

    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': ckpt}},
        '10': {'class_type': 'LoadImage', 'inputs': {'image': image_name}},
        '11': {'class_type': 'ImageScale', 'inputs': {'image': ['10',0], 'width': w, 'height': h, 'upscale_method': 'lanczos', 'crop': 'center'}},
        '12': {'class_type': 'VAEEncode', 'inputs': {'pixels': ['11',0], 'vae': ['1',2]}},
        '20': {'class_type': 'CLIPSetLastLayer', 'inputs': {'clip': ['1',1], 'stop_at_clip_layer': -2}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['20',0]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': neg_prompt, 'clip': ['20',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {
            'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0],
            'latent_image': ['12',0],
            'seed': int(time.time()*1000)%2**32,
            'steps': steps, 'cfg': cfg,
            'sampler_name': 'euler_ancestral', 'scheduler': 'normal',
            'denoise': strength,  # KEY: lower = closer to original
        }},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['1',2]}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'img2img'}}
    }

@app.post('/api/img2img')
async def img2img_endpoint(
    prompt: str = Form(...),
    model: str = Form('pony'),
    width: int = Form(832), height: int = Form(1216),
    steps: int = Form(25), cfg: float = Form(7.0),
    strength: float = Form(0.55),
    negative_prompt: str = Form(''),
    upscale: bool = Form(False),
    upscale_width: int = Form(1080), upscale_height: int = Form(1920),
    api_key: str = Form(...),
    image: UploadFile = File(...)
):
    """Image-to-Image: generate variation of input image guided by prompt"""
    ck(api_key)
    img_name = await upload_image(image)
    up = (upscale_width, upscale_height) if upscale else None
    job_id = uuid.uuid4().hex[:16]
    async def _run():
        wf = img2img_wf(img_name, prompt, model, width, height, steps, cfg, strength, negative_prompt)
        fp, err = await run_wf(wf)
        if err: raise Exception(err)
        if up: fp = await do_upscale_image(fp, up[0], up[1])
        return fp
    start_job(job_id, 'img2img', _run())
    return {'job_id': job_id, 'status': 'processing', 'type': 'img2img'}


# ════════════════════════════════════════════════════════════════
# ENDFRAME — Video with Start + End Image (WAN 2.2)
# ════════════════════════════════════════════════════════════════

@app.post('/api/startend-to-video')
async def startend_to_video(
    prompt: str = Form(...),
    model: str = Form('wan22'),
    frames: int = Form(81), steps: int = Form(25), cfg: float = Form(6.0),
    api_key: str = Form(...),
    start_image: UploadFile = File(...),
    end_image: UploadFile = File(...)
):
    """Generate video from start frame + end frame. WAN 2.2 interpolates between them."""
    ck(api_key)
    start_name = await upload_image(start_image)
    end_name = await upload_image(end_image)
    job_id = uuid.uuid4().hex[:16]
    async def _run():
        # Use WAN 2.2 with start+end conditioning
        # Build I2V workflow with end_image as additional conditioning
        if 'wan22' in model or 'wan' in model:
            wf = wan22_i2v(start_name, prompt, frames=frames, steps=steps, cfg=cfg)
            # Inject end frame conditioning if the model supports it
            # WAN 2.2 MoE supports this via the WanVideoWrapper node
            wf['90'] = {'class_type': 'LoadImage', 'inputs': {'image': end_name}}
            # Try to find the WanVideo node and add end_image
            for nid, node in wf.items():
                if 'WanVideo' in node.get('class_type', '') or 'Wan' in node.get('class_type', ''):
                    if 'end_image' in str(node):
                        node['inputs']['end_image'] = ['90', 0]
                        break
        else:
            # Fallback: just use start image I2V
            wf = wan22_i2v(start_name, prompt, frames=frames, steps=steps, cfg=cfg)

        fp, err = await run_wf(wf)
        if err: raise Exception(err)
        return fp
    start_job(job_id, 'startend-video', _run())
    return {'job_id': job_id, 'status': 'processing', 'type': 'startend-video'}


if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=5000)


## 9. API + Tunnel starten

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null
!chmod +x /usr/local/bin/cloudflared
!pip install fastapi uvicorn python-multipart httpx -q

import subprocess, threading, re, time, requests as _rq, os

# Alte Prozesse killen
subprocess.run(['pkill', '-9', '-f', 'api_server.py'], capture_output=True)
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# === Warten bis ComfyUI bereit ist ===
print('Warte auf ComfyUI...')
for i in range(60):
    try:
        r = _rq.get('http://127.0.0.1:8188/system_stats', timeout=3)
        if r.status_code == 200:
            print('ComfyUI bereit!')
            break
    except:
        pass
    time.sleep(5)
    if i % 6 == 5: print('  Noch warten... (' + str((i+1)*5) + 's)')
else:
    print('WARNUNG: ComfyUI antwortet nicht! Zelle 7 nochmal ausfuehren.')

# === API Server starten ===
api_proc = subprocess.Popen(['python', '/content/api_server.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(3)
print('API server gestartet (PID: ' + str(api_proc.pid) + ')')

# === Watchdog: Neustart bei Crash ===
def watchdog():
    global api_proc
    while True:
        time.sleep(10)
        if api_proc and api_proc.poll() is not None:
            print('[WATCHDOG] API Server gecrasht, Neustart in 3s...')
            time.sleep(3)
            subprocess.run(['pkill', '-9', '-f', 'api_server.py'], capture_output=True)
            time.sleep(1)
            api_proc = subprocess.Popen(['python', '/content/api_server.py'],
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            print('[WATCHDOG] API Server neugestartet (PID: ' + str(api_proc.pid) + ')')

wd = threading.Thread(target=watchdog, daemon=True)
wd.start()

# === Cloudflare Tunnel ===
tunnel_url = None

def start_tunnel():
    global tunnel_url
    proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5000'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for line in proc.stderr:
        m = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if m:
            tunnel_url = m.group(1)
            sep = '=' * 60
            print()
            print(sep)
            print('  CILLIAN AI STUDIO')
            print('  URL: ' + tunnel_url)
            print('  KEY: ironclaw-video-2026')
            print(sep)
            print()
            print('  VIDEO:')
            print('    POST /api/text-to-video   model=ltx|wan      ~2-4min')
            print('    POST /api/image-to-video  model=ltx|wan      ~2-4min')
            print('  BILDER:')
            print('    POST /api/text-to-image   model=flux|flux-schnell|sdxl|sd35  ~15-80s')
            print('  AUDIO:')
            print('    POST /api/speech-to-text  (Whisper)           ~5s')
            print('    POST /api/audio-ask       (Qwen2-Audio)       ~10s')
            print('    POST /api/text-to-speech  (Kokoro TTS)        ~2s')
            print()
            print('  GET /health')
            try:
                _rq.post('https://panel.cillianstudio.com/api/colab-url', params={'url': tunnel_url}, timeout=10)
                print('  Dashboard URL aktualisiert!')
            except:
                print('  Dashboard nicht erreichbar')
            break

t = threading.Thread(target=start_tunnel, daemon=True)
t.start()
time.sleep(15)
if not tunnel_url: print('Tunnel noch nicht bereit, warte...')

## 10. Testen

In [ ]:
# Quick Test: alle Endpoints
import requests, time

BASE = 'http://127.0.0.1:5000'

# Health
r = requests.get(BASE + '/health', timeout=5)
print('Health:', r.status_code)

# Image
r = requests.post(BASE + '/api/text-to-image',
    data={'prompt': 'red circle', 'model': 'flux-schnell', 'width': 256, 'height': 256, 'steps': 2, 'api_key': 'ironclaw-video-2026'})
print('Image submit:', r.json())

# TTS
r = requests.post(BASE + '/api/text-to-speech',
    data={'text': 'Hallo Welt', 'voice': 'af_heart', 'api_key': 'ironclaw-video-2026'})
print('TTS submit:', r.json())

print('Alle Endpoints OK!')

In [ ]:
# DEBUG: Check ComfyUI nodes + test simple workflow
import httpx, asyncio, json

async def debug():
    async with httpx.AsyncClient(timeout=30) as c:
        # Check available nodes
        r = await c.get('http://127.0.0.1:8188/object_info')
        nodes = list(r.json().keys())
        check = ['UnetLoaderGGUF','UNETLoader','EmptyLatentImage','KSampler',
                 'VAEDecode','SaveImage','DualCLIPLoader','CLIPLoader',
                 'VHS_VideoCombine','EmptyLTXVLatentVideo','EmptySD3LatentImage',
                 'CLIPVisionLoader','LTXVImgToVideo','unCLIPConditioning']
        print('=== Node Check ===')
        for n in check:
            print(f'  {n}: {"YES" if n in nodes else "NO"}')
        
        # Check models
        print('\n=== Models in unet/ ===')
        import os
        for d in ['unet','vae','clip','loras']:
            p = f'/content/ComfyUI/models/{d}'
            files = os.listdir(p) if os.path.exists(p) else []
            print(f'  {d}/: {files}')

        # Try simple prompt
        print('\n=== Test Simple Workflow ===')
        import time
        wf = {
            '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'flux1-schnell-Q4_K_S.gguf'}},
            '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': 256, 'height': 256, 'batch_size': 1}},
            '3': {'class_type': 'DualCLIPLoader', 'inputs': {'clip_name1': 'clip_l.safetensors', 'clip_name2': 't5xxl_fp8_e4m3fn.safetensors', 'type': 'flux'}},
            '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'a red circle', 'clip': ['3',0]}},
            '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['3',0]}},
            '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': 42, 'steps': 4, 'cfg': 1.0, 'sampler_name': 'euler', 'scheduler': 'simple', 'denoise': 1.0}},
            '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['8',0]}},
            '8': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ae.safetensors'}},
            '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'debug_test'}}
        }
        r = await c.post('http://127.0.0.1:8188/prompt', json={'prompt': wf})
        print(f'  Submit: {r.status_code}')
        if r.status_code == 200:
            pid = r.json().get('prompt_id')
            print(f'  Prompt ID: {pid}')
            print('  Waiting for result...')
        else:
            print(f'  Error: {r.text[:500]}')

await debug()


## 11. Keep Alive
Diese Zelle laufen lassen damit die Session aktiv bleibt.

In [ ]:
# Server Debug — zeigt genauen Crash-Grund
import subprocess, time

# Alten Server killen
subprocess.run(["pkill", "-9", "-f", "api_server"], capture_output=True)
time.sleep(2)

# Server starten und Output einfangen
proc = subprocess.Popen(
    ["python", "/content/api_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(5)

if proc.poll() is not None:
    print("SERVER GECRASHT!")
    print("STDOUT:", proc.stdout.read())
    print("STDERR:", proc.stderr.read())
else:
    print(f"Server laeuft (PID: {proc.pid})")
    import requests
    try:
        r = requests.get("http://127.0.0.1:5000/health", timeout=3)
        print(f"Health: {r.status_code} {r.text[:200]}")
    except Exception as e:
        print(f"Laeuft aber antwortet noch nicht: {e}")
        time.sleep(10)
        try:
            r = requests.get("http://127.0.0.1:5000/health", timeout=3)
            print(f"Health: {r.status_code} {r.text[:200]}")
        except Exception as e2:
            print(f"Antwortet immer noch nicht: {e2}")
            # Letzter Versuch: stderr lesen
            proc.kill()
            print("STDOUT:", proc.stdout.read()[:3000])
            print("STDERR:", proc.stderr.read()[:3000])


In [ ]:
import time, torch
print(f'Cillian AI Studio')
print(f'URL: {tunnel_url}')
print(f'Key: ironclaw-video-2026')
print(f'\n12 Funktionen: T2V, I2V, T2I, STT, Audio-Ask, TTS')
print(f'Laeuft... (Ctrl+M+I zum Stoppen)\n')
i = 0
while True:
    i += 1; time.sleep(300)
    vram = torch.cuda.memory_allocated()/1024**3
    print(f'{i*5}min | VRAM: {vram:.1f}GB | {tunnel_url}')